In [2]:
import os
import json

import pandas as pd
from sklearn.model_selection import GroupShuffleSplit

In [3]:
base_path = "/home/klymy/data/ObjectDetectionV3_2"
df = pd.read_csv(os.path.join(base_path, "annotation.csv"))
df_imgs = pd.read_csv(os.path.join(base_path, "images_grouped_th_0.65_holdout.csv"))
df_imgs.head()

/tmp/ipykernel_208663/700450693.py:2: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(os.path.join(base_path, "annotation.csv"))


,id,name,width,height,batch,folder,task_id,job_id,job_url,group_id
0,bdd640fb-0667-4ad1-9c80-317fa3b1799d,Cam2_H255_discaunt_29012024_2024-02-06-11h11m0...,640,480,annotation einav,einav1,653525,832182,https://app.cvat.ai/tasks/653525/jobs/832182,6e191042-370b-4063-9d90-e79eb888f6ed
1,b74d0fb1-32e7-4629-8fad-c1a606cb0fb3,Cam2_H255_discaunt_29012024_2024-02-06-11h11m1...,640,480,annotation einav,einav1,653525,832182,https://app.cvat.ai/tasks/653525/jobs/832182,6e191042-370b-4063-9d90-e79eb888f6ed
2,1a2a73ed-562b-4f79-8374-59eef50bea63,Cam2_H255_discaunt_29012024_2024-02-06-11h11m1...,640,480,annotation einav,einav1,653525,832182,https://app.cvat.ai/tasks/653525/jobs/832182,6e191042-370b-4063-9d90-e79eb888f6ed
3,e2acf72f-9e57-4f7a-a0ee-89aed453dd32,Cam2_H255_discaunt_29012024_2024-02-06-11h11m1...,640,480,annotation einav,einav1,653525,832182,https://app.cvat.ai/tasks/653525/jobs/832182,6e191042-370b-4063-9d90-e79eb888f6ed
4,29a3b2e9-5d65-4441-9588-42dea2bc372f,Cam2_H255_discaunt_29012024_2024-02-06-11h11m2...,640,480,annotation einav,einav1,653525,832182,https://app.cvat.ai/tasks/653525/jobs/832182,6e191042-370b-4063-9d90-e79eb888f6ed


In [4]:
print("before", df_imgs.shape, df.shape)

# TODO: do not remove image without annotations when problem with batch4_22.5/5 will be fixed
# rm images without annotations
tmp_df = pd.merge(df, df_imgs, how="outer", left_on="img_id", right_on="id")
tmp_ids = tmp_df[tmp_df["id_x"].isna()]["id_y"].values
df_imgs = df_imgs[~df_imgs["id"].isin(tmp_ids)]
print("number of images without annotations:", len(tmp_ids))

# rm images with problems in annotations
tmp_ids = df[(df["xtl"].isna()) | (df["x"].isna())]["img_id"].unique()
df_imgs = df_imgs[~df_imgs["id"].isin(tmp_ids)]
df = df[~df["img_id"].isin(tmp_ids)]
print("number of images with 'problematic' annotations:", len(tmp_ids))

#rm duplicates
df1 = df_imgs.groupby(['name','batch', "folder", "id"]).size().reset_index()[['name','batch', "folder", "id"]]
duplicates = df1[df1.duplicated('name', keep=False)]
tmp_ids = duplicates.iloc[1::2]["id"].values
df_imgs = df_imgs[~df_imgs["id"].isin(tmp_ids)]
df = df[~df["img_id"].isin(tmp_ids)]
print("number of duplicates:", len(tmp_ids))

df_imgs_holdout = df_imgs[df_imgs["group_id"] == "holdout"]
df_imgs = df_imgs[df_imgs["group_id"] != "holdout"]

print("after", df_imgs.shape, df.shape)

before (2065, 10) (62931, 16)
number of images without annotations: 7
number of images with 'problematic' annotations: 106
number of duplicates: 0
after (1916, 10) (58722, 16)


In [5]:
image_ids = df_imgs["id"].values
group_ids = df_imgs["group_id"].values
# labels = sorted(df["label"].unique())
labels = set([
    'bar_stool',
    'cabinet',
    'cabinet_FULL',
    'coffee_table',
    'coffee_table_FULL',
    'desk',
    'desk_FULL',
    'dinning_chair',
    'lounge_chair',
    'meeting_room_table',
    'meeting_room_table_FULL',
    'office_chair',
    'sofa',
])

val_size: float = 0.1
randon_seed: int = 44
num_skip_valid_splits: int = 0

n_splits = round(1 / val_size)
gskf = GroupShuffleSplit(n_splits=n_splits, random_state=randon_seed)
current_try = 0
# Add check for classes and groups
for index, indices in enumerate(gskf.split(image_ids, groups=group_ids)):
    print(f"Trying split # {index}")
    train_indexes, test_indexes = indices
    train_ids = image_ids[train_indexes]
    test_ids = image_ids[test_indexes]
    train_classes = sorted(df[df["img_id"].isin(train_ids)]["label"].unique())
    test_classes = sorted(df[df["img_id"].isin(test_ids)]["label"].unique())

    is_satisfactory_split = len(labels) == len(labels.intersection(train_classes)) and len(labels) == len(labels.intersection(test_classes))
    if is_satisfactory_split:
        if current_try >= num_skip_valid_splits:
            print("OK")
            break
        else:
            current_try += 1
    elif index == n_splits-1:
        print("not OK")
        break
    
print(f"train: {len(train_ids)} test: {len(test_ids)}")

with open(os.path.join(base_path, "train_test_split.json"), "w") as f:
    json.dump({"train": train_ids.tolist(), "test": test_ids.tolist()}, f)

Trying split # 0
OK
train: 1503 test: 413


In [6]:
with open(os.path.join(base_path, "holdout_split.json"), "w") as f:
    json.dump({"test": df_imgs_holdout["id"].values.tolist(), "train": []}, f)